# Patch-MLP-TS: Benchmark, Ablation, Efficiency
Research-grade notebook. Sections:
1. Setup & config
2. Data pipeline
3. Model definitions (Linear, NLinear, DLinear, PatchMLPTS, PatchTST, PatchMLPTS ablation variants)
4. Training / evaluation utilities
5. Main benchmark (ETT x4 + Electricity, multi-seed)
6. Ablation study
7. Efficiency comparison (params / train time / inference time)
8. Final result tables


## 1. Setup & Config

In [ ]:
import os
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
SEED = 42
BATCH_SIZE = 64
ELECTRICITY_BATCH = 16   # smaller batch, C~320 channels heavy in channel-independent reshape
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 5
LOOKBACK = 96

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("Seed:", SEED)

In [ ]:
# ---- EDIT THESE PATHS for your environment ----
DATA_DIR = "/kaggle/input/datasets/vedantjadhavv/electricity-data"

DATASETS = {
    "ETTh1": os.path.join(DATA_DIR, "ETTh1.csv"),
    "ETTh2": os.path.join(DATA_DIR, "ETTh2.csv"),
    "ETTm1": os.path.join(DATA_DIR, "ETTm1.csv"),
    "ETTm2": os.path.join(DATA_DIR, "ETTm2.csv"),
    "Electricity": os.path.join(DATA_DIR, "electricity.csv"),
}

for name, path in DATASETS.items():
    print(name, "->", path, "| exists:", os.path.exists(path))

## 2. Data Pipeline

In [ ]:
class TimeSeriesWindowDataset(Dataset):
    def __init__(self, data, lookback, horizon):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.lookback = lookback
        self.horizon = horizon
        self.length = len(data) - lookback - horizon + 1

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.lookback]
        y = self.data[idx + self.lookback: idx + self.lookback + self.horizon]
        return x, y

In [ ]:
def load_dataframe(path):
    df = pd.read_csv(path)
    first_col = df.columns[0]
    try:
        pd.to_datetime(df[first_col])
        df = df.drop(columns=[first_col])
    except Exception:
        pass

    df = df.select_dtypes(include=[np.number])
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(axis=1, how="all")
    df = df.interpolate(limit_direction="both")
    return df


def make_dataloaders(path, lookback, horizon, batch_size=64,
                      train_ratio=0.7, val_ratio=0.1, feature_limit=None):
    df = load_dataframe(path)
    if feature_limit is not None:
        df = df.iloc[:, :feature_limit]

    values = df.values.astype(np.float32)
    n = len(values)

    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_raw = values[:train_end]
    val_raw = values[train_end:val_end]
    test_raw = values[val_end:]

    mean = train_raw.mean(axis=0)
    std = train_raw.std(axis=0)
    std[std < 1e-6] = 1.0

    train_data = (train_raw - mean) / std
    val_data = (val_raw - mean) / std
    test_data = (test_raw - mean) / std

    train_dataset = TimeSeriesWindowDataset(train_data, lookback, horizon)
    val_dataset = TimeSeriesWindowDataset(val_data, lookback, horizon)
    test_dataset = TimeSeriesWindowDataset(test_data, lookback, horizon)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                               drop_last=False, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                             drop_last=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                              drop_last=False, num_workers=2, pin_memory=True)

    stats = {"mean": mean, "std": std, "num_features": values.shape[1], "total_rows": n}
    return train_loader, val_loader, test_loader, stats

In [ ]:
# sanity check
train_loader, val_loader, test_loader, stats = make_dataloaders(
    DATASETS["ETTh1"], lookback=96, horizon=96, batch_size=BATCH_SIZE
)
print("Features:", stats["num_features"])
print("Rows:", stats["total_rows"])
x, y = next(iter(train_loader))
print("X shape:", x.shape)
print("Y shape:", y.shape)

## 3. Model Definitions

All channel-independent models reshape `(B, L, C) -> (B*C, L)` for per-channel processing.
**Important:** permute to `(B, C, L)` before reshape — reshaping `(B, L, C)` directly to
`(B*C, L)` silently scrambles data (L and C get mixed since C is the innermost dim). This
was caught and fixed during development — verify any new model follows the same pattern.

In [ ]:
class LinearModel(nn.Module):
    """Single shared linear layer across all channels (LTSF-Linear baseline)."""
    def __init__(self, lookback, horizon, num_features):
        super().__init__()
        self.linear = nn.Linear(lookback, horizon)

    def forward(self, x):
        # x: [B, L, C]
        x = x.permute(0, 2, 1)      # [B, C, L]
        y = self.linear(x)          # [B, C, H]
        return y.permute(0, 2, 1)   # [B, H, C]


class NLinearModel(nn.Module):
    """Linear + last-value normalization (handles distribution shift)."""
    def __init__(self, lookback, horizon, num_features):
        super().__init__()
        self.linear = nn.Linear(lookback, horizon)

    def forward(self, x):
        last = x[:, -1:, :]         # [B, 1, C]
        x = x - last
        x = x.permute(0, 2, 1)      # [B, C, L]
        y = self.linear(x)          # [B, C, H]
        y = y.permute(0, 2, 1)      # [B, H, C]
        return y + last


class DLinearModel(nn.Module):
    """Trend/seasonal decomposition + two linear heads, shared across channels."""
    def __init__(self, lookback, horizon, num_features, moving_avg_kernel=25):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.moving_avg = nn.AvgPool1d(kernel_size=moving_avg_kernel, stride=1,
                                        padding=moving_avg_kernel // 2)
        self.seasonal_linear = nn.Linear(lookback, horizon)
        self.trend_linear = nn.Linear(lookback, horizon)

    def forward(self, x):
        B, L, C = x.shape
        x_bc = x.permute(0, 2, 1).reshape(B * C, L).unsqueeze(1)  # permute BEFORE reshape
        trend = self.moving_avg(x_bc).squeeze(1)
        seasonal = x_bc.squeeze(1) - trend
        seasonal_out = self.seasonal_linear(seasonal)
        trend_out = self.trend_linear(trend)
        pred = (seasonal_out + trend_out).reshape(B, C, self.horizon)
        return pred.permute(0, 2, 1)

In [ ]:
class PatchMLPTS(nn.Module):
    """Patch-based, channel-independent all-MLP forecaster (this paper's model)."""
    def __init__(self, lookback, horizon, num_features,
                 patch_len=12, embed_dim=128, num_blocks=3, dropout=0.1):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.patch_len = patch_len
        self.num_patches = (lookback - patch_len) // patch_len + 1

        self.patch_embed = nn.Linear(patch_len, embed_dim)
        self.mlp_blocks = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim * 4, embed_dim),
                nn.Dropout(dropout),
            ) for _ in range(num_blocks)
        ])
        self.head = nn.Linear(embed_dim, horizon)

    def forward(self, x):
        B, L, C = x.shape
        P, N = self.patch_len, self.num_patches
        L_eff = N * P
        x = x[:, :L_eff, :]
        x = x.reshape(B, N, P, C)
        x = x.permute(0, 3, 1, 2).reshape(B * C, N, P)  # channel-independent patches
        e = self.patch_embed(x)
        for block in self.mlp_blocks:
            e = e + block(e)
        z = e.mean(dim=1)
        y = self.head(z)
        return y.reshape(B, C, self.horizon).permute(0, 2, 1)


class PatchMLPTS_NoPatching(nn.Module):
    """Ablation: same MLP backbone, no patching -- embeds the whole lookback at once."""
    def __init__(self, lookback, horizon, num_features,
                 embed_dim=128, num_blocks=3, dropout=0.1):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.input_embed = nn.Linear(lookback, embed_dim)
        self.mlp_blocks = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim * 4, embed_dim),
                nn.Dropout(dropout),
            ) for _ in range(num_blocks)
        ])
        self.head = nn.Linear(embed_dim, horizon)

    def forward(self, x):
        B, L, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, L)  # channel-independent, whole series
        e = self.input_embed(x)
        for block in self.mlp_blocks:
            e = e + block(e)
        y = self.head(e)
        return y.reshape(B, C, self.horizon).permute(0, 2, 1)


class PatchMLPTS_ChannelMixing(nn.Module):
    """Ablation: same patching/MLP backbone, but channels mixed into each patch token
    instead of processed independently."""
    def __init__(self, lookback, horizon, num_features,
                 patch_len=12, embed_dim=128, num_blocks=3, dropout=0.1):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.num_features = num_features
        self.patch_len = patch_len
        self.num_patches = (lookback - patch_len) // patch_len + 1

        self.patch_embed = nn.Linear(patch_len * num_features, embed_dim)
        self.mlp_blocks = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim * 4, embed_dim),
                nn.Dropout(dropout),
            ) for _ in range(num_blocks)
        ])
        self.head = nn.Linear(embed_dim, horizon * num_features)

    def forward(self, x):
        B, L, C = x.shape
        P, N = self.patch_len, self.num_patches
        L_eff = N * P
        x = x[:, :L_eff, :]
        x = x.reshape(B, N, P, C)
        x = x.reshape(B, N, P * C)          # channels mixed into patch token
        e = self.patch_embed(x)
        for block in self.mlp_blocks:
            e = e + block(e)
        z = e.mean(dim=1)
        y = self.head(z)
        return y.reshape(B, self.horizon, C)

In [ ]:
class PatchTST(nn.Module):
    """Channel-independent patched Transformer encoder (Nie et al., 2023) --
    used here as the attention-based comparison point for PatchMLPTS."""
    def __init__(self, lookback, horizon, num_features,
                 patch_len=16, stride=8, d_model=128, n_heads=8,
                 num_layers=2, dropout=0.1, d_ff=256):
        super().__init__()
        self.lookback = lookback
        self.horizon = horizon
        self.patch_len = patch_len
        self.stride = stride
        self.num_patches = (lookback - patch_len) // stride + 2  # +1 padded patch

        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(self.num_patches * d_model, horizon)

    def forward(self, x):
        B, L, C = x.shape
        x = x.permute(0, 2, 1)                      # [B, C, L]
        pad = x[:, :, -1:].repeat(1, 1, self.stride)  # replicate-pad last value
        x = torch.cat([x, pad], dim=-1)              # [B, C, L+stride]
        x = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)  # [B, C, N, P]
        x = x.reshape(B * C, self.num_patches, self.patch_len)
        e = self.patch_embed(x) + self.pos_embed
        e = self.encoder(e)
        e = e.reshape(B * C, -1)
        y = self.head(e)
        return y.reshape(B, C, self.horizon).permute(0, 2, 1)

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


MAIN_MODELS = ["Linear", "NLinear", "DLinear", "PatchMLPTS", "PatchTST"]


def create_model(name, lookback, horizon, num_features):
    if name == "Linear":
        return LinearModel(lookback, horizon, num_features)
    if name == "NLinear":
        return NLinearModel(lookback, horizon, num_features)
    if name == "DLinear":
        return DLinearModel(lookback, horizon, num_features, moving_avg_kernel=25)
    if name == "PatchMLPTS":
        return PatchMLPTS(lookback, horizon, num_features,
                           patch_len=12, embed_dim=128, num_blocks=3, dropout=0.1)
    if name == "PatchMLPTS_NoPatching":
        return PatchMLPTS_NoPatching(lookback, horizon, num_features,
                                      embed_dim=128, num_blocks=3, dropout=0.1)
    if name == "PatchMLPTS_ChannelMixing":
        return PatchMLPTS_ChannelMixing(lookback, horizon, num_features,
                                         patch_len=12, embed_dim=128, num_blocks=3, dropout=0.1)
    if name == "PatchTST":
        return PatchTST(lookback, horizon, num_features,
                         patch_len=16, stride=8, d_model=128, n_heads=8,
                         num_layers=2, dropout=0.1)
    raise ValueError(f"unknown model: {name}")

## 4. Training / Evaluation Utilities

In [ ]:
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.sum = 0.0; self.count = 0
    def update(self, val, n=1): self.sum += val * n; self.count += n
    def avg(self): return self.sum / self.count if self.count > 0 else 0.0


def fit_model(model, train_loader, val_loader, test_loader,
              epochs=20, lr=1e-3, weight_decay=1e-4, patience=5,
              device="cuda", verbose=True):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=max(patience // 2, 1)
    )

    best_val = float("inf")
    best_state = None
    patience_counter = 0
    history = {"train_mse": [], "val_mse": []}

    for epoch in range(epochs):
        model.train()
        train_meter = AverageMeter()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            train_meter.update(loss.item(), x.size(0))

        model.eval()
        val_meter = AverageMeter()
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                val_meter.update(F.mse_loss(pred, y).item(), x.size(0))

        train_mse, val_mse = train_meter.avg(), val_meter.avg()
        history["train_mse"].append(train_mse)
        history["val_mse"].append(val_mse)
        scheduler.step(val_mse)

        if val_mse < best_val:
            best_val = val_mse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose:
            print(f"epoch {epoch+1:02d} | train {train_mse:.6f} | val {val_mse:.6f}")
        if patience_counter >= patience:
            if verbose: print("early stop")
            break

    model.load_state_dict(best_state)
    model.eval()
    test_mse_meter, test_mae_meter = AverageMeter(), AverageMeter()
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            test_mse_meter.update(F.mse_loss(pred, y).item(), x.size(0))
            test_mae_meter.update(F.l1_loss(pred, y).item(), x.size(0))

    return model, test_mse_meter.avg(), test_mae_meter.avg(), history

In [ ]:
def measure_epoch_time(model, loader, device, optimizer, n_epochs=3):
    model.train()
    criterion = nn.MSELoss()
    times = []
    for _ in range(n_epochs):
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
        if device.type == "cuda": torch.cuda.synchronize()
        times.append(time.time() - t0)
    return sum(times) / len(times)


def measure_inference_time(model, loader, device, n_batches=20):
    model.eval()
    times = []
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= n_batches:
                break
            x = x.to(device)
            if device.type == "cuda": torch.cuda.synchronize()
            t0 = time.time()
            _ = model(x)
            if device.type == "cuda": torch.cuda.synchronize()
            times.append(time.time() - t0)
    return (sum(times) / len(times)) * 1000  # ms/batch

## 5. Main Benchmark (ETT x4 + Electricity)

In [ ]:
ETT_DATASETS = ["ETTh1", "ETTh2", "ETTm1", "ETTm2"]
HORIZONS = [96, 192, 336, 720]

results = []

for dataset_name in ETT_DATASETS:
    path = DATASETS[dataset_name]
    for horizon in HORIZONS:
        print("\n" + "=" * 70)
        print(f"{dataset_name} | horizon={horizon}")
        print("=" * 70)

        train_loader, val_loader, test_loader, stats = make_dataloaders(
            path, lookback=LOOKBACK, horizon=horizon, batch_size=BATCH_SIZE
        )
        num_features = stats["num_features"]

        for model_name in MAIN_MODELS:
            print(f"\n--- {model_name} ---")
            seed_everything(SEED)
            model = create_model(model_name, LOOKBACK, horizon, num_features)
            params = count_parameters(model)

            model, test_mse, test_mae, history = fit_model(
                model, train_loader, val_loader, test_loader,
                epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
                patience=PATIENCE, device=DEVICE, verbose=False
            )

            print(f"{model_name}: MSE={test_mse:.4f} MAE={test_mae:.4f} params={params:,}")

            results.append({
                "seed": SEED, "dataset": dataset_name, "lookback": LOOKBACK,
                "horizon": horizon, "model": model_name,
                "parameters": params, "MSE": test_mse, "MAE": test_mae
            })

            del model
            torch.cuda.empty_cache()

df_ett_results = pd.DataFrame(results)
df_ett_results.to_csv("ett_benchmark_results.csv", index=False)
print("\nSaved ett_benchmark_results.csv")
df_ett_results

In [ ]:
ELECTRICITY_HORIZONS = [96, 720]  # 336/192 optional -- add to list if time allows

electricity_results = []
path = DATASETS["Electricity"]

for horizon in ELECTRICITY_HORIZONS:
    print("\n" + "=" * 70)
    print(f"Electricity | horizon={horizon}")
    print("=" * 70)

    train_loader, val_loader, test_loader, stats = make_dataloaders(
        path, lookback=LOOKBACK, horizon=horizon, batch_size=ELECTRICITY_BATCH
    )
    num_features = stats["num_features"]

    for model_name in MAIN_MODELS:
        print(f"\n--- {model_name} ---")
        seed_everything(SEED)
        model = create_model(model_name, LOOKBACK, horizon, num_features)
        params = count_parameters(model)

        model, test_mse, test_mae, history = fit_model(
            model, train_loader, val_loader, test_loader,
            epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
            patience=PATIENCE, device=DEVICE, verbose=False
        )

        print(f"{model_name}: MSE={test_mse:.4f} MAE={test_mae:.4f} params={params:,}")

        electricity_results.append({
            "seed": SEED, "dataset": "Electricity", "lookback": LOOKBACK,
            "horizon": horizon, "model": model_name,
            "parameters": params, "MSE": test_mse, "MAE": test_mae
        })

        del model
        torch.cuda.empty_cache()

df_electricity = pd.DataFrame(electricity_results)
df_electricity.to_csv("electricity_benchmark_results.csv", index=False)
print("\nSaved electricity_benchmark_results.csv")
df_electricity

### Multi-seed runs (optional, expensive)
Reruns the same grid with different seeds so the final table can report mean +/- std.
Scoped to H=96/720 only to keep runtime reasonable -- widen `GRID_HORIZONS` if you have budget.

In [ ]:
def run_full_grid(dataset_names, horizons, model_names, seeds,
                   batch_size=BATCH_SIZE, epochs=EPOCHS):
    all_results = []
    for seed in seeds:
        for dataset_name in dataset_names:
            path = DATASETS[dataset_name]
            bs = ELECTRICITY_BATCH if dataset_name == "Electricity" else batch_size
            for horizon in horizons:
                train_loader, val_loader, test_loader, stats = make_dataloaders(
                    path, lookback=LOOKBACK, horizon=horizon, batch_size=bs
                )
                num_features = stats["num_features"]
                for model_name in model_names:
                    seed_everything(seed)
                    model = create_model(model_name, LOOKBACK, horizon, num_features)
                    params = count_parameters(model)
                    model, test_mse, test_mae, history = fit_model(
                        model, train_loader, val_loader, test_loader,
                        epochs=epochs, lr=LR, weight_decay=WEIGHT_DECAY,
                        patience=PATIENCE, device=DEVICE, verbose=False
                    )
                    print(f"seed={seed} {dataset_name} H={horizon} {model_name}: MSE={test_mse:.4f}")
                    all_results.append({
                        "seed": seed, "dataset": dataset_name, "lookback": LOOKBACK,
                        "horizon": horizon, "model": model_name,
                        "parameters": params, "MSE": test_mse, "MAE": test_mae
                    })
                    del model
                    torch.cuda.empty_cache()
    return pd.DataFrame(all_results)

In [ ]:
# EXTRA_SEEDS = [43, 44]
# GRID_DATASETS = ["ETTh1", "ETTh2", "ETTm1", "ETTm2", "Electricity"]
# GRID_HORIZONS = [96, 720]
#
# df_extra_seeds = run_full_grid(GRID_DATASETS, GRID_HORIZONS, MAIN_MODELS, EXTRA_SEEDS)
# df_extra_seeds.to_csv("multiseed_extra_results.csv", index=False)
# df_extra_seeds

# Uncomment above to run seeds 43/44. Skipped by default -- heavy (100 fits).

In [ ]:
def summarize_seeds(df):
    return df.groupby(["dataset", "horizon", "model"]).agg(
        MSE_mean=("MSE", "mean"), MSE_std=("MSE", "std"),
        MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
        parameters=("parameters", "first"),
        n_seeds=("MSE", "count")
    ).reset_index()


# combine whatever seed runs exist so far (works with 1 seed too -- std will be NaN)
df_h96_720 = pd.concat([
    df_ett_results[df_ett_results["horizon"].isin([96, 720])],
    df_electricity
], ignore_index=True)

# if you ran the multi-seed cell above, uncomment:
# df_h96_720 = pd.concat([df_h96_720, df_extra_seeds], ignore_index=True)

df_main_summary = summarize_seeds(df_h96_720)
df_main_summary.to_csv("main_benchmark_summary.csv", index=False)
df_main_summary

## 6. Ablation Study
Full PatchMLPTS vs. no-patching vs. channel-mixing variants.

In [ ]:
ABLATION_DATASETS = ["ETTh1", "ETTh2"]
ABLATION_MODELS = ["PatchMLPTS", "PatchMLPTS_NoPatching", "PatchMLPTS_ChannelMixing"]

ablation_results = []

for dataset_name in ABLATION_DATASETS:
    path = DATASETS[dataset_name]
    for horizon in HORIZONS:
        print("\n" + "=" * 70)
        print(f"ABLATION | {dataset_name} | H={horizon}")
        print("=" * 70)

        train_loader, val_loader, test_loader, stats = make_dataloaders(
            path, lookback=LOOKBACK, horizon=horizon, batch_size=BATCH_SIZE
        )
        C = stats["num_features"]

        for model_name in ABLATION_MODELS:
            print(f"\n--- {model_name} ---")
            seed_everything(SEED)
            model = create_model(model_name, LOOKBACK, horizon, C)
            params = count_parameters(model)

            model, test_mse, test_mae, history = fit_model(
                model, train_loader, val_loader, test_loader,
                epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
                patience=PATIENCE, device=DEVICE, verbose=False
            )

            print(f"MSE={test_mse:.4f} MAE={test_mae:.4f}")

            ablation_results.append({
                "dataset": dataset_name, "lookback": LOOKBACK, "horizon": horizon,
                "model": model_name, "parameters": params, "MSE": test_mse, "MAE": test_mae
            })

            del model
            torch.cuda.empty_cache()

df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv("patchmlpts_ablation_results.csv", index=False)
print("\nSaved patchmlpts_ablation_results.csv")
df_ablation

In [ ]:
ablation_pivot = df_ablation.pivot_table(
    index=["dataset", "horizon"], columns="model", values=["MSE", "MAE"]
)
ablation_pivot.to_csv("ablation_summary_table.csv")
ablation_pivot

## 7. Efficiency Comparison: PatchMLPTS vs PatchTST
Params, training time per epoch, inference latency -- on a small-channel dataset (ETTh1)
and a large-channel dataset (Electricity) to show how each scales with C.

In [ ]:
efficiency_results = []
EFF_DATASETS = ["ETTh1", "Electricity"]
EFF_HORIZON = 96
EFF_MODELS = ["PatchMLPTS", "PatchTST"]

for dataset_name in EFF_DATASETS:
    path = DATASETS[dataset_name]
    batch = ELECTRICITY_BATCH if dataset_name == "Electricity" else BATCH_SIZE
    train_loader, val_loader, test_loader, stats = make_dataloaders(
        path, lookback=LOOKBACK, horizon=EFF_HORIZON, batch_size=batch
    )
    num_features = stats["num_features"]

    for model_name in EFF_MODELS:
        seed_everything(SEED)
        model = create_model(model_name, LOOKBACK, EFF_HORIZON, num_features).to(DEVICE)
        params = count_parameters(model)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        epoch_time = measure_epoch_time(model, train_loader, DEVICE, optimizer, n_epochs=3)
        inf_time = measure_inference_time(model, test_loader, DEVICE, n_batches=20)

        print(f"{dataset_name} | {model_name}: params={params:,} "
              f"epoch_time={epoch_time:.2f}s inference={inf_time:.2f}ms/batch")

        efficiency_results.append({
            "dataset": dataset_name, "model": model_name, "parameters": params,
            "epoch_time_sec": epoch_time, "inference_ms_per_batch": inf_time
        })
        del model
        torch.cuda.empty_cache()

df_efficiency = pd.DataFrame(efficiency_results)
df_efficiency.to_csv("efficiency_results.csv", index=False)
df_efficiency

## 8. Final Result Tables

In [ ]:
print("=== MAIN BENCHMARK (mean +/- std where multiple seeds available) ===")
display(df_main_summary)

print("\n=== ABLATION ===")
display(ablation_pivot)

print("\n=== EFFICIENCY (PatchMLPTS vs PatchTST) ===")
display(df_efficiency)

### Notes for write-up (keep claims scoped to what the data shows)
- Report **competitive accuracy + lower parameter count** vs PatchTST, not universal superiority
  over every baseline -- plain Linear/DLinear are often close or better on average MSE.
- Patching effect is **dataset/horizon-dependent**: check `ablation_pivot` before claiming
  patching universally helps (NoPatching sometimes wins).
- Channel-independence: claim benefit only where `ablation_pivot` shows ChannelMixing
  consistently worse -- check both ETTh1 and ETTh2 before generalizing.
- Any multi-seed claim (mean +/- std) is only valid for the (dataset, horizon) rows where
  `n_seeds` > 1 in `df_main_summary` -- note single-seed rows as such in tables.
